In [1]:
from optunaz.three_step_opt_build_merge import (
    optimize,
    buildconfig_best,
    build_best)

from optunaz.config import ModelMode, OptimizationDirection
from optunaz.config.optconfig import (
    OptimizationConfig,
    RandomForestClassifier,
    SVC,
    LogisticRegression,
    KNeighborsClassifier,
)
from optunaz.datareader import Dataset
from optunaz.descriptors import (ECFP,
                                 MACCS_keys,
                                 ECFP_counts,
                                 UnscaledPhyschemDescriptors,
                                 CompositeDescriptor,
                                 PhyschemDescriptors
                                 
)
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.metrics import confusion_matrix as cm, classification_report, matthews_corrcoef
from sklearn.metrics import PrecisionRecallDisplay, precision_recall_curve, auc as AUC
from optunaz.utils.preprocessing.splitter import Stratified

In [2]:
def model_b(train_file, test_file):

    config = OptimizationConfig(
        
        data = Dataset(
            input_column = 'Structure',
            response_column = 'Class',
            training_dataset_file=train_file,
            test_dataset_file=test_file,
        ),
        descriptors=[
            ECFP.new(),
            ECFP_counts.new(),
            MACCS_keys.new(),
            PhyschemDescriptors.new(),
            UnscaledPhyschemDescriptors.new(),
            CompositeDescriptor.new(
                descriptors=[
                    ECFP_counts.new(),
                    UnscaledPhyschemDescriptors.new(),
                ]
            ),
            CompositeDescriptor.new(
                descriptors=[
                    MACCS_keys.new(),
                    UnscaledPhyschemDescriptors.new(),
                ]
            ),
            CompositeDescriptor.new(
                descriptors=[
                    ECFP.new(),
                    UnscaledPhyschemDescriptors.new(),
                ]
            )
        ],
        algorithms=[
            RandomForestClassifier.new(),
            SVC.new(),
            LogisticRegression.new(),
            KNeighborsClassifier.new(),
        ],
        settings=OptimizationConfig.Settings(
            mode=ModelMode.CLASSIFICATION,
            cross_validation=10,
            cv_split_strategy=Stratified(),
            n_trials=200,
            n_startup_trials=50,
            direction=OptimizationDirection.MAXIMIZATION,
            random_seed=42,
            n_jobs=-1
        ),
    )

    # Run the optimization

    # study = optimize(config, study_name='study_RS')

    # build_best(buildconfig_best(study), r"M:\ML_scripts\MODEL DATA\RS_datasets\RS-models\RS_model_1512.pkl")

    # with open(r"M:\ML_scripts\MODEL DATA\RS_datasets\RS-models\RS_model_42.pkl", "rb") as f:
    #     model = pickle.load(f)

In [ ]:
import os
import pickle

def model_builder(train_file, test_file, output_model_path):
    config = OptimizationConfig(
        data=Dataset(
            input_column='Structure',
            response_column='Class',
            training_dataset_file=train_file,
            test_dataset_file=test_file,
        ),
        descriptors=[
            # ECFP.new(),
            ECFP_counts.new(radius=3, nBits=2048),
            # MACCS_keys.new(),
            # PhyschemDescriptors.new(),
            # UnscaledPhyschemDescriptors.new(),
            # CompositeDescriptor.new(
            #     descriptors=[
            #         ECFP_counts.new(),
            #         UnscaledPhyschemDescriptors.new(),
            #     ]
            # ),
            # CompositeDescriptor.new(
            #     descriptors=[
            #         MACCS_keys.new(),
            #         UnscaledPhyschemDescriptors.new(),
            #     ]
            # ),
            # CompositeDescriptor.new(
            #     descriptors=[
            #         ECFP.new(),
            #         UnscaledPhyschemDescriptors.new(),
            #     ]
            # )
        ],
        algorithms=[
            RandomForestClassifier.new(),
            # SVC.new(),
            # LogisticRegression.new(),
            # KNeighborsClassifier.new(),
        ],
        settings=OptimizationConfig.Settings(
            mode=ModelMode.CLASSIFICATION,
            cross_validation=10,
            cv_split_strategy=Stratified(),
            n_trials=200,
            n_startup_trials=50,
            direction=OptimizationDirection.MAXIMIZATION,
            random_seed=42,
            n_jobs=-1
        ),
    )

    study = optimize(config, study_name='study_RS')
    build_best(buildconfig_best(study), output_model_path)

    # Load the model

    with open(output_model_path, "rb") as f:
        model = pickle.load(f)

    # Load the test dataset

    test_data = pd.read_csv(config.data.test_dataset_file)

    test_smiles = test_data[config.data.input_column]
    y_true = test_data[config.data.response_column]

    # Predictions

    y_pred = model.predict_from_smiles(test_smiles)

    # ROC-AUC

    fpr, tpr, thresh = sklearn.metrics.roc_curve(y_true, y_pred)

    # Optimal threshold using ROC curve for binary conversion

    optimal_idx = np.argmin(np.sqrt(np.square(1-tpr) + np.square(fpr)))
    optimal_threshold_roc = thresh[optimal_idx]

    y_pred_bin = [1 if value >= optimal_threshold_roc else 0 for value in y_pred]

    mcc = matthews_corrcoef(y_pred_bin, y_true)
    auc = np.trapz(tpr, fpr)

    prec, rec, _ = precision_recall_curve(y_true, y_pred_bin)
    prc = np.trapz(rec, prec)

    # log the results/model info

    log_path = r"M:\ML_scripts\MODEL DATA\RS_datasets\RS_100_models\run_2_models\log2.txt"
    
    with open(log_path, "a") as log_file:
        log_file.write(f"Model: {model.metadata}\n")
        log_file.write(f"AUC: {auc}\n")
        log_file.write(f"MCC: {mcc}\n")
        log_file.write(f"AUPRC: {prc}\n")
        # log_file.write(f"Classification Report:\n{classification_report(y_true, y_pred_bin)}\n")
        log_file.write("\n")

    return 

def process_dataset_folders(root_folder, output_folder):
    for subdir, dirs, files in os.walk(root_folder):
        train_file = None
        test_file = None

        for file in files:
            if "train" in file.lower():
                train_file = os.path.join(subdir, file)
            elif "test" in file.lower():
                test_file = os.path.join(subdir, file)

        if train_file and test_file:
            model_name = os.path.basename(subdir) + "_model.pkl"
            output_model_path = os.path.join(output_folder, model_name)
            print(f"Processing: {subdir}")
            model_builder(train_file, test_file, output_model_path)


root_data_folder = r"M:\ML_scripts\MODEL DATA\RS_datasets\RS_100_sets"
output_models_folder = r"M:\ML_scripts\MODEL DATA\RS_datasets\RS_100_models\run_2_models"
process_dataset_folders(root_data_folder, output_models_folder)
